In [68]:
!pip install ortools

In [69]:
from ortools.linear_solver import pywraplp

In [70]:
tamanho_barra = int(input("Informe o tamanho da barra original (ex: 150): "))
qtd_tipos = int(input("Informe a quantidade de tipos de itens da demanda (ex: 3): "))

tamanhos_itens = []
demandas_itens = []

for i in range(qtd_tipos):
    tam = int(input(f"Tamanho do item {i+1}: "))
    dem = int(input(f"Demanda do item {i+1}: "))
    tamanhos_itens.append(tam)
    demandas_itens.append(dem)

In [71]:
print("\n" + "="*40)
print("GERANDO PADRÕES DE CORTE...")
print("="*40)

padroes = []
desperdicios = []


menor_peca = min(tamanhos_itens)

def gerar_padroes(idx_item, padrao_atual, tamanho_ocupado):

    if idx_item == qtd_tipos:
        desperdicio_atual = tamanho_barra - tamanho_ocupado


        if sum(padrao_atual) > 0 and desperdicio_atual < menor_peca:
            padroes.append(padrao_atual)
            desperdicios.append(desperdicio_atual)
        return

    max_qtd_possivel = (tamanho_barra - tamanho_ocupado) // tamanhos_itens[idx_item]

    for qtd in range(max_qtd_possivel + 1):
        novo_tamanho = tamanho_ocupado + (qtd * tamanhos_itens[idx_item])
        gerar_padroes(idx_item + 1, padrao_atual + [qtd], novo_tamanho)

gerar_padroes(0, [], 0)

print("\n" + "="*70)
print("PADRÕES DE CORTE GERADOS")
print("="*70)

print(f"{'Padrão':<10} {'Composição do corte':<30} {'Vetor':<15} {'Sobra'}")
print("-"*70)

for i, p in enumerate(padroes):

    descricao = []

    for j in range(qtd_tipos):
        if p[j] > 0:
            descricao.append(f"{p[j]}x{tamanhos_itens[j]}")

    descricao_final = " + ".join(descricao)

    print(
        f"{i+1:<10} "
        f"{descricao_final:<30} "
        f"{str(p):<15} "
        f"{desperdicios[i]}"
    )


GERANDO PADRÕES DE CORTE...

PADRÕES DE CORTE GERADOS
Padrão     Composição do corte            Vetor           Sobra
----------------------------------------------------------------------
1          12x500                         [0, 0, 0, 0, 12] 0
2          1x850 + 10x500                 [0, 0, 0, 1, 10] 150
3          2x850 + 8x500                  [0, 0, 0, 2, 8] 300
4          3x850 + 6x500                  [0, 0, 0, 3, 6] 450
5          4x850 + 5x500                  [0, 0, 0, 4, 5] 100
6          5x850 + 3x500                  [0, 0, 0, 5, 3] 250
7          6x850 + 1x500                  [0, 0, 0, 6, 1] 400
8          7x850                          [0, 0, 0, 7, 0] 50
9          1x1200 + 9x500                 [0, 0, 1, 0, 9] 300
10         1x1200 + 1x850 + 7x500         [0, 0, 1, 1, 7] 450
11         1x1200 + 2x850 + 6x500         [0, 0, 1, 2, 6] 100
12         1x1200 + 3x850 + 4x500         [0, 0, 1, 3, 4] 250
13         1x1200 + 4x850 + 2x500         [0, 0, 1, 4, 2] 400
14   

In [72]:
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

x = {}
for i in range(len(padroes)):
    x[i] = solver.IntVar(0, infinity, f"x_{i+1}")

# Função objetivo:
# minimizar a quantidade total de barras usadas
objetivo = solver.Objective()
for i in range(len(padroes)):
    objetivo.SetCoefficient(x[i], 1)

objetivo.SetMinimization()

# Restrições de demanda
for j in range(qtd_tipos):
    restricao = solver.Constraint(demandas_itens[j], infinity, f"Demanda_Tamanho_{tamanhos_itens[j]}")
    for i in range(len(padroes)):
        restricao.SetCoefficient(x[i], padroes[i][j])

In [73]:
print("\n" + "="*70)
print("MODELO DE PROGRAMAÇÃO LINEAR INTEIRA")
print("="*70)
print(solver.ExportModelAsLpFormat(False))

print("\n" + "="*70)
print("RESULTADO DA OTIMIZAÇÃO")
print("="*70)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:

    print("SOLUÇÃO ÓTIMA ENCONTRADA!\n")
    print(f"Valor da função objetivo: {int(objetivo.Value())}")

    print("\n" + "="*100)
    print("QUANTIDADE A CORTAR DE CADA PADRÃO")
    print("="*100)

    total_barras_usadas = 0
    desperdicio_total_real = 0

    print(
        f"{'Padrão':<10} "
        f"{'Corte de cada barra':<35} "
        f"{'Qtd. a cortar':<15} "
        f"{'Desperdício unit.':<20} "
        f"{'Desperdício total'}"
    )
    print("-"*100)

    for i in range(len(padroes)):

        qtd_usada = int(x[i].solution_value())

        if qtd_usada > 0:

            total_barras_usadas += qtd_usada
            desperdicio_padrao = qtd_usada * desperdicios[i]
            desperdicio_total_real += desperdicio_padrao

            descricao = []

            for j in range(qtd_tipos):
                if padroes[i][j] > 0:
                    descricao.append(f"{padroes[i][j]} peça(s) de {tamanhos_itens[j]}m")

            descricao_final = " + ".join(descricao)

            print(
                f"{i+1:<10} "
                f"{descricao_final:<35} "
                f"{qtd_usada:<15} "
                f"{desperdicios[i]:<20} "
                f"{desperdicio_padrao}"
            )

    print("\n" + "="*100)
    print(f"TOTAL DE BARRAS ORIGINAIS UTILIZADAS : {total_barras_usadas}")
    print(f"DESPERDÍCIO TOTAL FINAL              : {desperdicio_total_real}")
    print("="*100)


MODELO DE PROGRAMAÇÃO LINEAR INTEIRA
\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 5
\   Variables        : 57
\     Binary         : 0
\     Integer        : 57
\     Continuous     : 0
Minimize
 Obj: +1 x_1 +1 x_2 +1 x_3 +1 x_4 +1 x_5 +1 x_6 +1 x_7 +1 x_8 +1 x_9 +1 x_10 +1 x_11 +1 x_12 +1 x_13 +1 x_14 +1 x_15 +1 x_16 +1 x_17 +1 x_18 +1 x_19 +1 x_20 +1 x_21 +1 x_22 +1 x_23 +1 x_24 +1 x_25 +1 x_26 +1 x_27 +1 x_28 +1 x_29 +1 x_30 +1 x_31 +1 x_32 +1 x_33 +1 x_34 +1 x_35 +1 x_36 +1 x_37 +1 x_38 +1 x_39 +1 x_40 +1 x_41 +1 x_42 +1 x_43 +1 x_44 +1 x_45 +1 x_46 +1 x_47 +1 x_48 +1 x_49 +1 x_50 +1 x_51 +1 x_52 +1 x_53 +1 x_54 +1 x_55 +1 x_56 +1 x_57 
Subject to
 Demanda_Tamanho_2350: +1 x_41 +1 x_42 +1 x_43 +1 x_44 +1 x_45 +1 x_46 +1 x_47 +1 x_48 +1 x_49 +1 x_50 +1 x_51 +1 x_52 +1 x_53 +1 x_54 +2 x_55 +2 x_56 +2 x_57  >= 8
 Demanda_Tamanho_2250: +1 x_26 +1 x_27 +1 x_28 +1 x_29 +1 x_30 +1 x_31 +1 x_32 +1 x_33 +1 x_34 +1 x_35 +1 x_